In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import pandas as pd
import pickle

In [ ]:
#Load the Trained (ANN) model
model= load_model('model.h5')

#Load the encoder and scaler:
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [ ]:
# Example input data (to do the prediction)
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

Now, in the above input data, convert categorical variables into numerical, then convert all the numerical data using standard scaler (using the scalers stored in pickeles)

1. Convert Cat into num
2. Apply standard scaling
3. Do prediction wrt the model


In [13]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [15]:
input_df= pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [ ]:
#Encode categorical variables
input_df['Gender']= label_encoder_gender.transform(input_df['Gender'])

In [20]:
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [21]:
#Concatination one hot encoded

input_df = pd.concat([input_df.drop('Geography', axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
#Scaling the input data
input_scaled = scaler.transform(input_df)


In [23]:
input_scaled


array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [24]:
#Prediction

prediction =model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step


array([[0.05558914]], dtype=float32)

In [25]:
prediction_proba = prediction[0][0]

In [26]:
prediction_proba

np.float32(0.055589143)

In [27]:
if prediction_proba >0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.


Stepwise:
1. Load all pickle files
2. When we get input data, convert it into dataframe, then convert categorical into numerical using labelencoder/onehot encoder, then combine everything(concat) and create a new dataframe, use scaling(transform feature)
3. Do the prediction and check the probability
 